# Yearly pairwise agreement
The API performs aggregation and export; this notebook presents the persisted result.

In [ ]:
from datetime import date
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from liljegren_analysis.comparison import (
    STATISTIC_SPECS, analyze_yearly_pairwise, pairwise_matrices,
)
from liljegren_analysis.comparison.config import REFERENCE_IMPLEMENTATION

REPOSITORY = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
INPUT_ROOT = REPOSITORY / 'data/pairwise/6-targets'
OUTPUT_ROOT = REPOSITORY / 'data/pairwise/yearly/2025'
START_DATE = date(2025, 1, 1)
END_DATE = date(2025, 12, 31)
SAMPLE_SCOPE = 'all-implementations'
REFERENCE = REFERENCE_IMPLEMENTATION
EXPORT_IMPLEMENTATIONS = (
    'heatstress', 'meteor', 'thermofeel',
    'kwodzicki-pywbgt', 'kong-huber-pywbgt',
)
DEFAULT_QUANTITY = 'wbgt'
DEFAULT_STATISTIC = 'mean_difference_c'

In [ ]:
OUTPUTS = analyze_yearly_pairwise(
    INPUT_ROOT, start_date=START_DATE, end_date=END_DATE,
    output_root=OUTPUT_ROOT, sample_scope=SAMPLE_SCOPE,
    reference_implementation=REFERENCE,
    export_implementations=EXPORT_IMPLEMENTATIONS,
)
YEARLY = pd.read_parquet(OUTPUTS.pairwise)
COVERAGE = pd.read_parquet(OUTPUTS.quantity_coverage)
MATRICES = pairwise_matrices(YEARLY)
display(COVERAGE)
OUTPUTS

## Interactive comparison matrix

In [ ]:
def show_matrix(quantity, statistic):
    matrix = MATRICES[quantity][statistic]
    signed = STATISTIC_SPECS[statistic]['signed']
    figure = go.Figure(go.Heatmap(
        z=matrix.to_numpy(), x=matrix.columns, y=matrix.index,
        text=matrix.map(lambda value: '' if pd.isna(value) else f'{value:,.2f}'),
        texttemplate='%{text}', colorscale='RdBu_r' if signed else 'YlOrRd',
        zmid=0 if signed else None,
        hovertemplate='<b>%{y} − %{x}</b><br>%{z:,.3f}<extra></extra>',
    ))
    figure.update_layout(
        template='plotly_dark', title=f'{quantity.upper()}: {statistic}', height=760,
        xaxis={'title': 'implementation_b'},
        yaxis={'title': 'implementation_a', 'autorange': 'reversed', 'scaleanchor': 'x'},
    )
    figure.show()

QUANTITY_CONTROL = widgets.Dropdown(options=tuple(MATRICES), value=DEFAULT_QUANTITY, description='Quantity')
STATISTIC_CONTROL = widgets.Dropdown(options=tuple(STATISTIC_SPECS), value=DEFAULT_STATISTIC, description='Statistic')
display(widgets.HBox([QUANTITY_CONTROL, STATISTIC_CONTROL]))
display(widgets.interactive_output(show_matrix, {'quantity': QUANTITY_CONTROL, 'statistic': STATISTIC_CONTROL}))